# Forçages Dynamiques et Interpolation Temporelle

Ce notebook démontre comment utiliser des forçages externes qui varient dans le temps.

**Concepts clés :**
- Séparation entre `initial_state` (variables d'état internes) et `forcings` (variables externes)
- Interpolation temporelle automatique des forçages
- Impact des conditions environnementales variables sur la dynamique biologique

**Scenario :**
- Biomasse de thon influencée par la température de l'eau
- Température fournie comme forçage externe (données satellitaires simulées)
- Température varie de façon saisonnière

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationController, SimulationConfig
from seapopym.standard import Coordinates

print("✅ Imports réussis")

## 1. Modèle Scientifique

### Équation de croissance thermiquement dépendante

$$\frac{dB}{dt} = g(T) \cdot B - m \cdot B$$

Où :
- $B$ : Biomasse (kg/m²)
- $T$ : Température (°C)
- $g(T) = g_{max} \cdot \exp\left(-\frac{(T - T_{opt})^2}{2\sigma^2}\right)$ : Taux de croissance thermique (courbe gaussienne)
- $m$ : Taux de mortalité constant

**Paramètres :**
- $g_{max} = 0.5$ jour⁻¹ : Croissance maximale
- $T_{opt} = 20°C$ : Température optimale
- $\sigma = 5°C$ : Largeur de la courbe thermique
- $m = 0.1$ jour⁻¹ : Mortalité

In [ ]:
# Paramètres du modèle
G_MAX = 0.5 / 86400  # Convertir jour⁻¹ en s⁻¹
T_OPT = 20.0  # °C
SIGMA = 5.0   # °C
MORTALITY = 0.1 / 86400  # s⁻¹

def thermal_growth(temperature, biomass):
    """Croissance thermiquement dépendante (gaussienne)."""
    # Courbe de performance thermique
    growth_factor = G_MAX * np.exp(-((temperature - T_OPT)**2) / (2 * SIGMA**2))
    growth_rate = growth_factor * biomass
    return {"growth": growth_rate}

def natural_mortality(biomass):
    """Mortalité naturelle constante."""
    return {"mortality": -MORTALITY * biomass}

print("✅ Fonctions du modèle définies")

## 2. Configuration du Blueprint

**Points importants :**
- `temperature` est déclarée comme **forcing** → fournie de l'extérieur
- `biomass` est déclarée comme **forcing** mais sera dans `initial_state` → variable d'état

In [ ]:
def configure_thermal_model(bp: Blueprint):
    """Configure le modèle thermiquement dépendant."""
    
    # Variables
    bp.register_forcing("temperature")  # Externe (forçage)
    bp.register_forcing("biomass")       # État interne
    
    # Groupe Tuna
    bp.register_group("Tuna", [
        {
            "func": thermal_growth,
            "output_mapping": {"growth": "growth_rate"},
            "output_tendencies": {"growth": "biomass"},
        },
        {
            "func": natural_mortality,
            "output_mapping": {"mortality": "mortality_rate"},
            "output_tendencies": {"mortality": "biomass"},
        }
    ])

print("✅ Configuration du Blueprint créée")

In [ ]:
# Visualisation du graphe
bp = Blueprint()
configure_thermal_model(bp)
plan = bp.build()

fig = bp.visualize(title="Modèle de Croissance Thermiquement Dépendante")
plt.show()

print(f"\nVariables initiales : {plan.initial_variables}")
print(f"Variables produites : {plan.produced_variables}")

## 3. Création des Forçages Dynamiques

### Scénario : Cycle saisonnier de température

On simule 365 jours avec des données de température tous les 30 jours (~12 points).
Le `ForcingManager` interpolera automatiquement entre ces points.

**Point important :** Les forçages doivent couvrir **toute la période de simulation** 
(du `start_date` au `end_date` inclus), sinon le `ForcingManager` lèvera une erreur 
lorsqu'il tentera d'accéder à un temps hors limites.

In [ ]:
# Configuration temporelle
start_date = datetime(2020, 1, 1)
end_date = datetime(2020, 12, 31)
timestep = timedelta(days=1)

# Données de température : points tous les 30 jours
# IMPORTANT : On doit s'assurer que les forçages couvrent toute la période de simulation
forcing_times = pd.date_range(start_date, end_date, freq="30D")

# Si end_date n'est pas inclus, on l'ajoute explicitement
if forcing_times[-1] < pd.Timestamp(end_date):
    forcing_times = forcing_times.append(pd.DatetimeIndex([end_date]))

print(f"Nombre de points de forçage : {len(forcing_times)}")
print(f"Premier point : {forcing_times[0]}")
print(f"Dernier point : {forcing_times[-1]}")

# Température sinusoïdale : min=10°C (hiver), max=25°C (été), optimum=20°C
day_of_year = np.array([(t - start_date).days for t in forcing_times])
temperature_values = 17.5 + 7.5 * np.sin(2 * np.pi * (day_of_year - 80) / 365)

# Dataset de forçages
forcings = xr.Dataset(
    {
        "temperature": ((Coordinates.T, "x"), temperature_values[:, None]),
    },
    coords={
        Coordinates.T: forcing_times,
        "x": [0]
    }
)

print(f"\nForçages créés :")
print(f"  Période : {forcings[Coordinates.T].values[0]} → {forcings[Coordinates.T].values[-1]}")
print(f"  Temp min : {temperature_values.min():.1f}°C")
print(f"  Temp max : {temperature_values.max():.1f}°C")
print(f"\n✅ Forçages créés avec couverture complète")

In [ ]:
# Visualisation du cycle de température
plt.figure(figsize=(10, 4))
plt.plot(forcing_times, temperature_values, 'o-', linewidth=2, markersize=8, label='Points de forçage')
plt.axhline(T_OPT, color='green', linestyle='--', label=f'T optimale ({T_OPT}°C)')
plt.axhspan(T_OPT - SIGMA, T_OPT + SIGMA, alpha=0.2, color='green', label='Zone favorable (±σ)')
plt.xlabel('Date', fontsize=11)
plt.ylabel('Température (°C)', fontsize=11)
plt.title('Cycle Saisonnier de Température', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. État Initial

**Important :** Seule la biomasse est dans `initial_state`.
La température viendra des `forcings`.

In [ ]:
# État initial : biomasse uniquement
initial_state = xr.Dataset(
    {
        "biomass": (("x"), [0]),  # 50 kg/m²
    },
    coords={"x": [0]}
)

print("État initial :")
print(initial_state)
print("\n✅ État initial créé")

## 5. Configuration et Exécution de la Simulation

In [ ]:
# Configuration
config = SimulationConfig(
    start_date=start_date,
    end_date=end_date,
    timestep=timestep
)

# Setup du controller AVEC forcings
controller = SimulationController(config)
controller.setup(configure_thermal_model, initial_state, forcings=forcings)

print("Setup terminé.")
print(f"Forcing Manager actif : {controller.forcing_manager is not None}")
print("\n✅ Controller configuré avec forçages dynamiques")

In [ ]:
# Contrainte de positivité
controller.time_integrator.positive_vars = ["biomass"]

# Stockage des résultats
results = {
    "days": [],
    "dates": [],
    "biomass": [],
    "temperature": []
}

# Run la simulation pas à pas
current_day = 0
print("Simulation en cours...", end="")

while controller._current_time < config.end_date:
    controller.step()
    controller._current_time += config.timestep
    current_day += 1
    
    # Stockage (tous les 7 jours pour alléger)
    if current_day % 7 == 0:
        results["days"].append(current_day)
        results["dates"].append(controller._current_time)
        results["biomass"].append(controller.state["biomass"].item())
        results["temperature"].append(controller.state["temperature"].item())
        print(".", end="")

print(f"\n✅ Simulation terminée : {current_day} jours")

# Recalculer les taux à partir des valeurs stockées
# Note: growth_rate et mortality_rate ne sont pas stockés dans le state car ce sont des tendances
results["growth_rate"] = []
results["mortality_rate"] = []

for temp, bio in zip(results["temperature"], results["biomass"]):
    # Recalculer le taux de croissance
    growth_factor = G_MAX * np.exp(-((temp - T_OPT)**2) / (2 * SIGMA**2))
    results["growth_rate"].append(growth_factor * bio)
    
    # Recalculer le taux de mortalité
    results["mortality_rate"].append(-MORTALITY * bio)

print(f"✅ Taux recalculés (longueurs : temp={len(results['temperature'])}, bio={len(results['biomass'])}, rates={len(results['growth_rate'])})")

### Note technique : Variables de tendance

**Pourquoi recalculer les taux ?**

Les variables `growth_rate` et `mortality_rate` sont des **tendances** (définies via `output_tendencies` dans le Blueprint). Elles sont utilisées par l'intégrateur temporel pour mettre à jour `biomass`, mais ne sont **pas conservées** dans le state final.

**Flux de calcul :**
1. Les unités calculent `growth_rate` et `mortality_rate`
2. Le `TimeIntegrator` les utilise pour calculer : `biomass(t+dt) = biomass(t) + (growth_rate + mortality_rate) * dt`
3. Seul `biomass` (et les diagnostics) sont conservés dans le state

Pour l'analyse, on recalcule donc ces taux à partir de `biomass` et `temperature` stockées.

## 6. Visualisation des Résultats

### Observation attendue :
- La biomasse croît davantage quand T ≈ 20°C (optimal)
- La croissance ralentit quand T s'éloigne de l'optimum

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Subplot 1 : Température interpolée
ax1 = axes[0]
ax1.plot(results["dates"], results["temperature"], 'o-', color='red', linewidth=2, markersize=4, label='Temp. interpolée')
ax1.plot(forcing_times, temperature_values, 's', markersize=10, color='darkred', alpha=0.6, label='Points de forçage')
ax1.axhline(T_OPT, color='green', linestyle='--', alpha=0.7, label=f'T optimale ({T_OPT}°C)')
ax1.axhspan(T_OPT - SIGMA, T_OPT + SIGMA, alpha=0.15, color='green')
ax1.set_ylabel('Température (°C)', fontsize=11)
ax1.set_title('Forçage Dynamique : Température avec Interpolation Temporelle', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Subplot 2 : Biomasse
ax2 = axes[1]
ax2.plot(results["dates"], results["biomass"], 'o-', color='blue', linewidth=2, markersize=4)
ax2.set_ylabel('Biomasse (kg/m²)', fontsize=11)
ax2.set_title('Évolution de la Biomasse', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Subplot 3 : Taux de croissance et mortalité
ax3 = axes[2]
# Convertir en jour⁻¹ pour lisibilité
growth_per_day = np.array(results["growth_rate"]) * 86400
mortality_per_day = np.array(results["mortality_rate"]) * 86400
net_rate = growth_per_day + mortality_per_day

ax3.plot(results["dates"], growth_per_day, 'o-', color='green', linewidth=2, markersize=3, label='Croissance')
ax3.plot(results["dates"], mortality_per_day, 'o-', color='red', linewidth=2, markersize=3, label='Mortalité')
ax3.plot(results["dates"], net_rate, 'o-', color='purple', linewidth=2, markersize=3, label='Taux net')
ax3.axhline(0, color='black', linestyle='-', alpha=0.3)
ax3.set_ylabel('Taux (jour⁻¹)', fontsize=11)
ax3.set_xlabel('Date', fontsize=11)
ax3.set_title('Taux de Croissance et Mortalité', fontsize=13, fontweight='bold')
ax3.legend(loc='upper right', fontsize=9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Analyse : Impact de la Température sur la Croissance

In [ ]:
# Courbe de performance thermique
temp_range = np.linspace(5, 35, 100)
growth_factor = G_MAX * 86400 * np.exp(-((temp_range - T_OPT)**2) / (2 * SIGMA**2))

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(temp_range, growth_factor, linewidth=3, color='green')
plt.axvline(T_OPT, color='red', linestyle='--', alpha=0.7, label=f'T opt = {T_OPT}°C')
plt.axvspan(T_OPT - SIGMA, T_OPT + SIGMA, alpha=0.2, color='green', label='±σ')
plt.scatter(results["temperature"], 
           np.array(results["growth_rate"]) * 86400 / np.array(results["biomass"]),
           c=results["days"], cmap='viridis', alpha=0.6, s=30, label='Simulation')
plt.xlabel('Température (°C)', fontsize=11)
plt.ylabel('Taux de croissance (jour⁻¹)', fontsize=11)
plt.title('Courbe de Performance Thermique', fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.colorbar(label='Jour')

plt.subplot(1, 2, 2)
plt.scatter(results["temperature"], results["biomass"], 
           c=results["days"], cmap='viridis', s=50, alpha=0.7)
plt.axvline(T_OPT, color='red', linestyle='--', alpha=0.7, label=f'T opt')
plt.xlabel('Température (°C)', fontsize=11)
plt.ylabel('Biomasse (kg/m²)', fontsize=11)
plt.title('Biomasse vs Température', fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.colorbar(label='Jour')

plt.tight_layout()
plt.show()

## 8. Statistiques Finales

In [ ]:
print("="*60)
print("STATISTIQUES DE LA SIMULATION")
print("="*60)

print(f"\n📅 Période : {start_date.date()} → {end_date.date()} ({current_day} jours)")

print(f"\n🌡️  Température :")
print(f"  - Min : {min(results['temperature']):.1f}°C")
print(f"  - Max : {max(results['temperature']):.1f}°C")
print(f"  - Optimale : {T_OPT}°C")

print(f"\n🐟 Biomasse :")
print(f"  - Initiale : {results['biomass'][0]:.2f} kg/m²")
print(f"  - Finale : {results['biomass'][-1]:.2f} kg/m²")
print(f"  - Variation : {results['biomass'][-1] - results['biomass'][0]:+.2f} kg/m²")
print(f"  - Facteur de croissance : {results['biomass'][-1] / results['biomass'][0]:.2f}x")

# Identifier les périodes favorables
favorable_periods = [(d, t) for d, t in zip(results["days"], results["temperature"]) 
                     if abs(t - T_OPT) < SIGMA]
print(f"\n📊 Conditions favorables (T ∈ [{T_OPT-SIGMA:.1f}, {T_OPT+SIGMA:.1f}]°C) :")
print(f"  - Jours : {len(favorable_periods)}/{len(results['days'])} échantillonnés")
print(f"  - Pourcentage : {100*len(favorable_periods)/len(results['days']):.1f}%")

print("\n" + "="*60)
print("✅ Démonstration terminée")

## 9. Points Clés de Cette Démonstration

### ✅ Ce que nous avons démontré :

1. **Séparation claire des responsabilités**
   - `initial_state` contient uniquement les variables d'état internes (biomasse)
   - `forcings` contient les variables externes (température)

2. **Interpolation temporelle automatique**
   - Données de forçage tous les 30 jours (12 points)
   - Simulation quotidienne (365 pas de temps)
   - Le `ForcingManager` interpole automatiquement la température pour chaque jour

3. **Validation stricte**
   - Erreur si une variable est définie à la fois dans `initial_state` et `forcings`
   - Erreur si des variables requises sont manquantes

4. **Impact des conditions environnementales**
   - La croissance est maximale quand T ≈ 20°C
   - La biomasse évolue différemment selon les saisons

### 🎯 Cas d'usage typiques :
- Données satellitaires (température, chlorophylle, courants)
- Sorties de modèles climatiques
- Observations in-situ
- Scénarios de changement climatique